In [1]:
import sqlite3 
import pandas as pd
import datetime as dt
import pyarrow as pa
import json


## Inspect and Clean the data


create a context manager that will handle opening, commiting changes and closing the database whith sqlite3

In [2]:
## class based approch, after further reasearch its is better to go with a fucntion based context manager,:
##      - its only a simple task
##       - the class creates an unnecessary instance objectin memory
# class DBContextManager:
#     def __init__(self, data_base):
#         print("establishing connection")
#         self.data_base = data_base
#     def __enter__(self):
#         try:
#             self.connection = sqlite3.connect(f'{self.databse}')
#         print(f"connected to {self.data_base}")  
        
#     def __exit__(self, exc_type, exc_value, traceback):
#         print(f"{exc_type}\n{exc_value}\n{traceback}")
#         self.connection.commit()
#         print(f"changes have been commited to {self.data_base}")
#         self.connection.close()
#         print(f"connection with {data_base} has been closed")
        

In [3]:
#function based contect manager
from contextlib import contextmanager

@contextmanager
def data_base_sqlite3_context_manager(data_base):
    print("establishing connection...")
    connection = sqlite3.connect(data_base)
    print(f"Successfully connected to {data_base}")
    try:
        yield connection

    #commit changes 
        connection.commit()
        print(f"Action(s) commited successfully to {data_base}")
    # handle errors etc
    except Exception as exc:
        print(f"Whoops, somethings gone tits up, lets have a little sticky beak: {exc}")
        connection.rollback()
        raise exc
    finally:
        connection.close()
        print(f"Connection to {data_base} has been closed")

In [37]:
#establish connection 
with data_base_sqlite3_context_manager('cademycode.db') as db_conn:
    #find table names 
    df = pd.read_sql_query("SELECT * FROM sqlite_master WHERE type = 'table';", db_conn)
    #upload tbales as database
df
    

establishing connection...
Successfully connected to cademycode.db
Action(s) commited successfully to cademycode.db
Connection to cademycode.db has been closed


,type,name,tbl_name,rootpage,sql
0,table,cademycode_students,cademycode_students,2,CREATE TABLE cademycode_students (\n\tuuid INT...
1,table,cademycode_courses,cademycode_courses,5,CREATE TABLE cademycode_courses (\n\tcareer_pa...
2,table,cademycode_student_jobs,cademycode_student_jobs,6,CREATE TABLE cademycode_student_jobs (\n\tjob_...


In [5]:
dfs.keys()

dict_keys(['cademycode_students', 'cademycode_courses', 'cademycode_student_jobs'])

In [6]:
##inspect first table cademycode_students
cademycode_students_df = dfs['cademycode_students']
cademycode_courses_df = dfs['cademycode_courses']
cademycode_student_jobs_df =dfs['cademycode_student_jobs']

### Student Data

See how many rows and columns there are respectivley

In [7]:
cademycode_students_df.shape

(5000, 9)

drop any duplicates and check again

In [8]:
cademycode_students_df.drop_duplicates()

,uuid,name,dob,sex,contact_info,job_id,num_course_taken,current_career_path_id,time_spent_hrs
0,1,Annabelle Avery,1943-07-03,F,"{""mailing_address"": ""303 N Timber Key, Irondal...",7.0,6.0,1.0,4.99
1,2,Micah Rubio,1991-02-07,M,"{""mailing_address"": ""767 Crescent Fair, Shoals...",7.0,5.0,8.0,4.4
2,3,Hosea Dale,1989-12-07,M,"{""mailing_address"": ""P.O. Box 41269, St. Bonav...",7.0,8.0,8.0,6.74
3,4,Mariann Kirk,1988-07-31,F,"{""mailing_address"": ""517 SE Wintergreen Isle, ...",6.0,7.0,9.0,12.31
4,5,Lucio Alexander,1963-08-31,M,"{""mailing_address"": ""18 Cinder Cliff, Doyles b...",7.0,14.0,3.0,5.64
...,...,...,...,...,...,...,...,...,...
4995,4996,Quentin van Harn,1967-07-07,N,"{""mailing_address"": ""591 Blue Berry, Coulee, I...",5.0,5.0,2.0,13.82
4996,4997,Alejandro van der Sluijs,1964-11-03,M,"{""mailing_address"": ""30 Iron Divide, Pewaukee ...",4.0,13.0,1.0,7.86
4997,4998,Brock Mckenzie,2004-11-25,M,"{""mailing_address"": ""684 Rustic Rest Avenue, C...",8.0,10.0,3.0,12.1
4998,4999,Donnetta Dillard,1943-02-12,N,"{""mailing_address"": ""900 Indian Oval, Euclid, ...",3.0,6.0,5.0,14.86


None of the rows were duplicates as there are still 5000 rows, Have a sticky beak at the data generally. 

In [9]:
cademycode_students_df.head(5)

,uuid,name,dob,sex,contact_info,job_id,num_course_taken,current_career_path_id,time_spent_hrs
0,1,Annabelle Avery,1943-07-03,F,"{""mailing_address"": ""303 N Timber Key, Irondal...",7.0,6.0,1.0,4.99
1,2,Micah Rubio,1991-02-07,M,"{""mailing_address"": ""767 Crescent Fair, Shoals...",7.0,5.0,8.0,4.4
2,3,Hosea Dale,1989-12-07,M,"{""mailing_address"": ""P.O. Box 41269, St. Bonav...",7.0,8.0,8.0,6.74
3,4,Mariann Kirk,1988-07-31,F,"{""mailing_address"": ""517 SE Wintergreen Isle, ...",6.0,7.0,9.0,12.31
4,5,Lucio Alexander,1963-08-31,M,"{""mailing_address"": ""18 Cinder Cliff, Doyles b...",7.0,14.0,3.0,5.64


take a look at the columns and thier types


In [10]:
cademycode_students_df.dtypes

uuid                       int64
name                      object
dob                       object
sex                       object
contact_info              object
job_id                    object
num_course_taken          object
current_career_path_id    object
time_spent_hrs            object
dtype: object

All data types are strings, to use `.describe()` they must be of a appropriate data types:
- dob `date`
- job_id `int`
- num_course_taken `int`
- current_career_path_id `int`
- time_spend_hrs `time`

In [11]:
#dob to datetime object
cademycode_students_df["dob"] = pd.to_datetime(cademycode_students_df["dob"], format="%Y-%m-%d")
#job_id, num_courses_taken, current_career_path_id to int object, add options of cuaing invalud enties to turn to NaN , keeping the type as usgined as id will always be a positive ineger,anf pyarrow to deal with nan values 
for col in ["job_id", "num_course_taken", "current_career_path_id"]:
    cademycode_students_df[col] = pd.to_numeric(cademycode_students_df[col], errors='coerce', downcast='unsigned', dtype_backend='pyarrow')
#time_spend_hrs to datetime object, will convert to minutes and save as integer to improve clarity and reduce RAM usage
cademycode_students_df["time_spent_mins"] = (pd.to_numeric(cademycode_students_df["time_spent_hrs"], dtype_backend='pyarrow') * 60).round().astype('Int64')
del cademycode_students_df["time_spent_hrs"]
cademycode_students_df.dtypes

uuid                               int64
name                              object
dob                       datetime64[ns]
sex                               object
contact_info                      object
job_id                    uint8[pyarrow]
num_course_taken          uint8[pyarrow]
current_career_path_id    uint8[pyarrow]
time_spent_mins                    Int64
dtype: object

look at the column statistics 

In [12]:
cademycode_students_df.time_spent_mins

0       299
1       264
2       404
3       739
4       338
       ... 
4995    829
4996    472
4997    726
4998    892
4999    602
Name: time_spent_mins, Length: 5000, dtype: Int64

The contact info columns needs pivoting, inspect it further


In [13]:
cademycode_students_df.contact_info.dtype


dtype('O')

In [14]:
cademycode_students_df.contact_info[1]

'{"mailing_address": "767 Crescent Fair, Shoals, Indiana, 37439", "email": "rubio6772@hmail.com"}'

looks like the rows contain an email and a mailing address these need to be their own seperate columns 

In [15]:
# new_contact_info = cademycode_students_df.contact_info.apply(lambda x: json.loads(x))
# new_contact_info_df = pd.json_normalize(new_contact_info)
# cademycode_students_df = pd.concat([cademycode_students_df.drop(columns=["contact_info"]), new_contact_info_df], axis=1)

In [16]:
def safe_json_loading(val):
    try:
        return json.loads(val)
    except (json.JSONDecodeError, ValueError, TypeError):
        return {}
new_contact_info = cademycode_students_df.contact_info.apply(safe_json_loading)
new_contact_info_df = pd.json_normalize(new_contact_info)
cademycode_students_df = pd.concat([cademycode_students_df.drop(columns=["contact_info"]), new_contact_info_df], axis=1)

In [17]:
cademycode_students_df.head()

,uuid,name,dob,sex,job_id,num_course_taken,current_career_path_id,time_spent_mins,mailing_address,email
0,1,Annabelle Avery,1943-07-03,F,7,6,1,299,"303 N Timber Key, Irondale, Wisconsin, 84736",annabelle_avery9376@woohoo.com
1,2,Micah Rubio,1991-02-07,M,7,5,8,264,"767 Crescent Fair, Shoals, Indiana, 37439",rubio6772@hmail.com
2,3,Hosea Dale,1989-12-07,M,7,8,8,404,"P.O. Box 41269, St. Bonaventure, Virginia, 83637",hosea_dale8084@coldmail.com
3,4,Mariann Kirk,1988-07-31,F,6,7,9,739,"517 SE Wintergreen Isle, Lane, Arkansas, 82242",kirk4005@hmail.com
4,5,Lucio Alexander,1963-08-31,M,7,14,3,338,"18 Cinder Cliff, Doyles borough, Rhode Island,...",alexander9810@hmail.com


In [18]:
cademycode_students_df.dtypes

uuid                               int64
name                              object
dob                       datetime64[ns]
sex                               object
job_id                    uint8[pyarrow]
num_course_taken          uint8[pyarrow]
current_career_path_id    uint8[pyarrow]
time_spent_mins                    Int64
mailing_address                   object
email                             object
dtype: object

In [19]:
cademycode_students_df.sex.value_counts()

sex
M    1995
F    1990
N    1015
Name: count, dtype: int64

In [20]:
cademycode_students_df.num_course_taken.value_counts()

num_course_taken
5     341
12    332
2     312
15    309
10    306
7     303
13    297
0     296
8     291
11    289
4     285
6     282
14    280
3     279
1     279
9     268
Name: count, dtype: int64[pyarrow]

In [21]:
cademycode_students_df.job_id.value_counts()

job_id
2    706
1    693
7    680
3    675
4    671
5    660
6    657
8    253
Name: count, dtype: int64[pyarrow]

In [22]:
cademycode_students_df.current_career_path_id.value_counts()

current_career_path_id
5     476
3     469
10    460
1     459
6     454
2     450
7     449
9     441
8     437
4     434
Name: count, dtype: int64[pyarrow]

In [23]:
cademycode_students_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   uuid                    5000 non-null   int64         
 1   name                    5000 non-null   object        
 2   dob                     5000 non-null   datetime64[ns]
 3   sex                     5000 non-null   object        
 4   job_id                  4995 non-null   uint8[pyarrow]
 5   num_course_taken        4749 non-null   uint8[pyarrow]
 6   current_career_path_id  4529 non-null   uint8[pyarrow]
 7   time_spent_mins         4529 non-null   Int64         
 8   mailing_address         5000 non-null   object        
 9   email                   5000 non-null   object        
dtypes: Int64(1), datetime64[ns](1), int64(1), object(4), uint8[pyarrow](3)
memory usage: 294.9+ KB


### Course Data

In [24]:
cademycode_courses_df.shape


(10, 3)

In [25]:
cademycode_courses_df =  cademycode_courses_df.drop_duplicates()

In [26]:
cademycode_courses_df

,career_path_id,career_path_name,hours_to_complete
0,1,data scientist,20
1,2,data engineer,20
2,3,data analyst,12
3,4,software engineering,25
4,5,backend engineer,18
5,6,frontend engineer,20
6,7,iOS developer,27
7,8,android developer,27
8,9,machine learning engineer,35
9,10,ux/ui designer,15


### Job Data

In [27]:
cademycode_student_jobs_df.isna()

,job_id,job_category,avg_salary
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
5,False,False,False
6,False,False,False
7,False,False,False
8,False,False,False
9,False,False,False


In [28]:
cademycode_student_jobs_df.shape

(13, 3)

In [29]:
cademycode_student_jobs_df = cademycode_student_jobs_df.drop_duplicates()
cademycode_student_jobs_df

,job_id,job_category,avg_salary
0,1,analytics,86000
1,2,engineer,101000
2,3,software developer,110000
3,4,creative,66000
4,5,financial services,135000
5,6,education,61000
6,7,HR,80000
7,8,student,10000
8,9,healthcare,120000
9,0,other,80000


### Create the output CSV 

merge all tables into one create a single csv 


In [30]:
cademycode_merged_df = cademycode_students_df.merge(cademycode_courses_df, how='left', left_on='current_career_path_id', right_on='career_path_id').drop(columns=["career_path_id"]).merge(cademycode_student_jobs_df, how='left', on='job_id')

In [31]:
cademycode_merged_df.dtypes

uuid                               int64
name                              object
dob                       datetime64[ns]
sex                               object
job_id                    uint8[pyarrow]
num_course_taken          uint8[pyarrow]
current_career_path_id    uint8[pyarrow]
time_spent_mins                    Int64
mailing_address                   object
email                             object
career_path_name                  object
hours_to_complete                float64
job_category                      object
avg_salary                       float64
dtype: object

In [32]:
cademycode_merged_df.shape

(5000, 14)

save as a csv

In [33]:
cademycode_merged_df.to_csv("cademycode_merged_tables.csv") #this pandas method has a built in context manager

create a database to upload the new table


In [34]:
# clean sqlite database (contex manager), upload cleaned dataframes as tables
with data_base_sqlite3_context_manager('analytics_ready_cademycode_data_base') as db_conn:
    data_frames = [cademycode_students_df, cademycode_courses_df, cademycode_student_jobs_df]
    for data_frame in data_frames:
        var_name = next(name for name, val in locals().items() if val is data_frame)
        table_name = var_name.removesuffix("_df")
        data_frame.to_sql(table_name, db_conn, if_exists="replace", index=False)
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", db_conn)  
    print(f"Tables created: \n{tables}")

establishing connection...
Successfully connected to analytics_ready_cademycode_data_base
Tables created: 
                  name
0  cademycode_students
1   cademycode_courses
2                  ___
Action(s) commited successfully to analytics_ready_cademycode_data_base
Connection to analytics_ready_cademycode_data_base has been closed


In [35]:
# #create a csv from the cleaned database
# with data_base_sqlite3_context_manager('analytics_ready_cademycode_data_base') as db_conn:
#     db_conn.cursor().execute("""CREATE TABLE analytics_ready_cademycode_table AS
#             SELECT uuid,
#                 s.name,
#                 s.dob,
#                 s.sex,
#                 s.job_id,
#                 s.num_course_taken,
#                 s.current_career_path_id,
#                 s.time_spent_mins,
#                 s.mailing_address,
#                 s.email,
#                 c.career_path_name ,
#                 c.hours_to_complete ,
#                 j.job_category ,
#                 j.avg_salary
#             FROM cademycode_students s 
#             LEFT JOIN cademycode_courses c
#             ON s.current_career_path_id = c.career_path_id
#             LEFT JOIN cademycode_student_jobs j
#             ON s.job_id = j.job_id;""")

In [36]:
#upload analytics table as csv
import csv
with data_base_sqlite3_context_manager('analytics_ready_cademycode_data_base') as db_conn:
    upload_data = db_conn.cursor().execute("SELECT * FROM analytics_ready_cademycode_table").fetchall
    with open('analytics_ready_cademycode_data.csv', 'w') as upload:
        writer = csv.writer(upload)


establishing connection...
Successfully connected to analytics_ready_cademycode_data_base
Whoops, somethings gone tits up, lets have a little sticky beak: no such table: analytics_ready_cademycode_table
Connection to analytics_ready_cademycode_data_base has been closed


OperationalError: no such table: analytics_ready_cademycode_table

In [ ]:
# context manager function that uploads the table from the database and saves it as a csv
@contextmanager
def table_to_csv(data_base, table_name, csv_name):
    """
    Uses a context manager 'data_base_sqlite3_context_manager' to connect to the database fetching all upload data.
    Sends upload data to a csv
    Args: data base name, csv filename
    """
    with data_base_sqlite3_context_manager(f'{data_base}') as db_conn:
        upload_data = db_conn.cursor().execute(f"SELECT * FROM {table_name}").fetchall
    csv_file = open(csv_name, 'write')
    try:
        yield csv_file
        upload = csv.writer(csv_file)
        upload.writerows(upload_data)
    except Exception as exc:
        raise exc
    finally:
        print("Table successfully uploaded as a CSV")